# Phase 3 + Phase 4 — chạy trọn gói

Chạy tuần tự từ trên xuống. Tổng khoảng **40–55 phút** trên T4.

### Đã sửa gì so với lần chạy trước

| # | Sửa ở đâu | Nội dung |
|---|---|---|
| 1 | `gnn.py` | `BatchNorm` → `LayerNorm`. Ở chế độ transductive, mỗi lượt truyền tiến chạy trên cả 8.030 node nên BatchNorm tính thống kê trên cả node validation — vừa bất ổn vừa là rò rỉ phân phối |
| 2 | `gnn.py` | **Early stopping**: tách 15% bệnh nhân trong tập train làm tập theo dõi. Trước đây chạy cứng 300 epoch nên GAT bị quá khớp và bị chấm oan so với MLP |
| 3 | `gnn.py` | GAT được nhận thêm **khối "điểm nguy cơ hàng xóm"** — thứ đã thắng ở B3b (+0,022 ROC, P=99%) |
| 4 | cả hai | Thêm `--out-dir`, và file tải về giờ gói thành **một file .zip** |

### Bảng kết quả sẽ có 4 cấu hình GAT

| Cấu hình | Trả lời câu hỏi |
|---|---|
| GAT đầy đủ | Kết quả chính |
| GAT `--no-graph` | Đồ thị đóng góp bao nhiêu (tắt truyền tin, giữ nguyên kiến trúc) |
| GAT `--no-nb-score` | Khối điểm hàng xóm đóng góp bao nhiêu |
| GAT trên `frozen` | Kết luận có phụ thuộc bộ đặc trưng ảnh không |

## 1. Bật GPU

**Runtime → Change runtime type → T4 GPU → Save**

In [ ]:
import torch
print("Torch:", torch.__version__)
print("GPU  :", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "KHÔNG THẤY GPU — vào Runtime → Change runtime type → T4 GPU")

## 2. Tải mã nguồn và dữ liệu

In [ ]:
import os, shutil
if os.path.exists("/content/repo"):
    shutil.rmtree("/content/repo")
!git clone -q -b fusion_gnn https://github.com/Do-Trung-Quan/DO-AN-TOT-NGHIEP-E22-T02.git /content/repo
os.chdir("/content/repo")

need = ["fusion/output/fusion_dataset_ketqua_control.npz",
        "fusion/output/fusion_dataset_ketqua_frozen.npz",
        "fusion/output/graph_ketqua_k10_bridge.npz",
        "fusion/output/tabular_ketqua.npz",
        "output/ketqua/ts_predictions.parquet"]
missing = [f for f in need if not os.path.exists(f)]
print("THIẾU:", missing) if missing else print("✅ Đủ file đầu vào")

## 3. Phase 3 — baseline cho cả hai nguồn ảnh

Khoảng 10–15 phút.

In [ ]:
!python fusion/baselines.py --source control --seeds 3 --epochs 60
!python fusion/baselines.py --source frozen  --seeds 3 --epochs 60

In [ ]:
from IPython.display import Markdown, display
display(Markdown(open("fusion/output/phase3_baselines_control.md", encoding="utf-8").read()))

## 4. Phase 4 — GAT đầy đủ, nguồn `control` (kết quả chính)

Khoảng 8–12 phút.

In [ ]:
!python fusion/gnn.py --source control --seeds 3 --epochs 300

In [ ]:
from IPython.display import Markdown, display
display(Markdown(open("fusion/output/phase4_gat_control.md", encoding="utf-8").read()))

## 5. Ablation A — tắt truyền tin trên đồ thị

Cùng kiến trúc, cùng tham số, chỉ khác là node **không nói chuyện với hàng xóm**.
Chênh lệch giữa cell 4 và cell này là đóng góp thuần của đồ thị.

In [ ]:
!python fusion/gnn.py --source control --seeds 3 --epochs 300 --no-graph

In [ ]:
from IPython.display import Markdown, display
display(Markdown(open("fusion/output/phase4_gat_nograph_control.md", encoding="utf-8").read()))

## 6. Ablation B — bỏ khối "điểm nguy cơ hàng xóm"

GAT vẫn truyền tin trên đồ thị, nhưng không được cho sẵn điểm nguy cơ của hàng xóm.
So với cell 4 để biết khối này đóng góp bao nhiêu.

In [ ]:
!python fusion/gnn.py --source control --seeds 3 --epochs 300 --no-nb-score

In [ ]:
from IPython.display import Markdown, display
display(Markdown(open("fusion/output/phase4_gat_nonb_control.md", encoding="utf-8").read()))

## 7. GAT trên nguồn `frozen`

Kiểm tra kết luận có đổi khi thay bộ đặc trưng ảnh không.

In [ ]:
!python fusion/gnn.py --source frozen --seeds 3 --epochs 300

In [ ]:
from IPython.display import Markdown, display
display(Markdown(open("fusion/output/phase4_gat_frozen.md", encoding="utf-8").read()))

## 8. Tải toàn bộ kết quả về — **một file zip duy nhất**

Tải từng file một hay bị trình duyệt chặn sau vài file (đó là lý do lần trước thiếu
file ablation). Gói thành một zip thì chắc chắn đủ.

Giải nén vào `fusion/output/` ở repo local rồi commit. **Không `git push` từ Colab.**

In [ ]:
import glob, zipfile, os
files = sorted(glob.glob("fusion/output/phase3_*") + glob.glob("fusion/output/phase4_*"))
with zipfile.ZipFile("/content/ketqua_phase3_4.zip", "w", zipfile.ZIP_DEFLATED) as z:
    for f in files:
        z.write(f, os.path.basename(f))
print(f"Đã gói {len(files)} file:")
for f in files:
    print("  -", os.path.basename(f))

from google.colab import files as colab_files
colab_files.download("/content/ketqua_phase3_4.zip")